# OpenAQ — Raw Data Acquisition

**OpenAQ Location ID:** `2848864`  
**Requested period:** 2024-05-16 through 2026-10-19

This notebook stops after creating and verifying a master raw CSV.


In [1]:
# Step 0 — Imports and configuration
import subprocess
import re
from pathlib import Path
import pandas as pd
import gzip

In [29]:
LOCATION_ID = 2848864
START_DATE = pd.Timestamp("2024-05-16")
END_DATE = pd.Timestamp("2026-10-19")

S3_BASE = (
    f"s3://openaq-data-archive/"
    f"records/csv.gz/locationid={LOCATION_ID}/"
)

PROJECT_DIR = Path("Dataset")
RAW_DIR = PROJECT_DIR / "raw"
TEMP_DIR = RAW_DIR / "s3_files_hadigaun_latest"
OUTPUT_FILE = PROJECT_DIR / "raw_data_hadigaun_latest.csv"

TEMP_DIR.mkdir(parents=True, exist_ok=True)

print("OpenAQ location:", LOCATION_ID)
print("Requested period:", START_DATE.date(), "to", END_DATE.date())
print("Project folder:", PROJECT_DIR.resolve())

OpenAQ location: 2848864
Requested period: 2024-05-16 to 2026-10-19
Project folder: D:\MSDSA\4th Sem\Thesis\Codes\Dataset


## Check AWS CLI

The notebook uses the AWS CLI with `--no-sign-request` because the OpenAQ archive is publicly accessible.

In [3]:
# Check AWS CLI
try:
    result = subprocess.run(
        ["aws", "--version"],
        capture_output=True,
        text=True,
        check=True
    )
    print(result.stdout.strip() or result.stderr.strip())
    print("AWS CLI is available.")
except FileNotFoundError:
    raise RuntimeError(
        "AWS CLI was not found. Install AWS CLI and restart Jupyter."
    )
except subprocess.CalledProcessError as e:
    raise RuntimeError(f"AWS CLI check failed: {e.stderr}")


aws-cli/2.37.9 Python/3.14.6 Windows/11 exe/AMD64
AWS CLI is available.


## Identify available S3 files

We ask S3 for all objects under the Dabali location, then keep only daily files dated from 2024-05-16 through 2026-10-19.

We do not assume every calendar date exists.

In [30]:
# List objects in the public OpenAQ archive
result = subprocess.run(
    ["aws", "s3", "ls", "--no-sign-request", "--recursive", S3_BASE],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    raise RuntimeError(
        "Could not access OpenAQ S3 archive:\n" + result.stderr
    )

s3_lines = result.stdout.splitlines()
print(f"Objects returned for this location: {len(s3_lines):,}")


Objects returned for this location: 736


See if the daily files are there or not

In [31]:
# Parse daily filenames and filter by requested date range
date_pattern = re.compile(r"location-\d+-(\d{8})\.csv\.gz$")

objects = []

for line in s3_lines:
    parts = line.split(maxsplit=3)
    if len(parts) < 4:
        continue

    s3_key = parts[3]
    match = date_pattern.search(s3_key)
    if not match:
        continue

    file_date = pd.to_datetime(match.group(1), format="%Y%m%d")

    if START_DATE <= file_date <= END_DATE:
        objects.append({"date": file_date, "s3_key": s3_key})

file_manifest = (
    pd.DataFrame(objects)
    .sort_values("date")
    .drop_duplicates("s3_key")
    .reset_index(drop=True)
)

if file_manifest.empty:
    raise RuntimeError("No matching files found.")

print(f"Matching daily files: {len(file_manifest):,}")
print("First:", file_manifest["date"].min().date())
print("Last :", file_manifest["date"].max().date())

display(file_manifest.head())


Matching daily files: 736
First: 2024-05-16
Last : 2026-10-19


,date,s3_key
0,2024-05-16,records/csv.gz/locationid=2848864/year=2024/mo...
1,2024-05-17,records/csv.gz/locationid=2848864/year=2024/mo...
2,2024-05-18,records/csv.gz/locationid=2848864/year=2024/mo...
3,2024-05-19,records/csv.gz/locationid=2848864/year=2024/mo...
4,2024-05-20,records/csv.gz/locationid=2848864/year=2024/mo...


Check for missing Dates

In [32]:
# Check missing calendar dates
expected_dates = pd.date_range(START_DATE, END_DATE, freq="D")
available_dates = pd.DatetimeIndex(file_manifest["date"].drop_duplicates())
missing_dates = expected_dates.difference(available_dates)

print(f"Expected calendar days: {len(expected_dates):,}")
print(f"Available file days : {len(available_dates):,}")
print(f"Missing file days   : {len(missing_dates):,}")

if len(missing_dates):
    print("First missing dates:")
    print(missing_dates[:30].strftime("%Y-%m-%d").tolist())

if len(missing_dates):
    print("Last missing dates:")
    print(missing_dates[-30:].strftime("%Y-%m-%d").tolist())


Expected calendar days: 887
Available file days : 736
Missing file days   : 151
First missing dates:
['2024-05-29', '2024-10-23', '2024-10-24', '2024-11-15', '2024-11-16', '2024-11-17', '2025-01-15', '2025-01-16', '2025-01-17', '2025-01-18', '2025-01-19', '2025-01-20', '2025-01-21', '2025-01-22', '2025-01-23', '2025-01-24', '2025-01-25', '2025-01-26', '2025-01-27', '2025-01-28', '2025-01-29', '2025-01-30', '2025-01-31', '2025-02-01', '2025-02-02', '2025-02-03', '2025-02-04', '2025-02-05', '2025-02-06', '2025-02-07']
Last missing dates:
['2025-05-13', '2025-05-14', '2025-05-15', '2025-05-16', '2025-05-17', '2025-05-18', '2025-05-19', '2025-05-20', '2025-05-21', '2025-05-22', '2025-06-10', '2025-06-11', '2025-06-12', '2025-06-13', '2025-06-14', '2025-06-15', '2025-06-16', '2025-06-17', '2025-06-18', '2025-06-19', '2025-06-20', '2025-06-21', '2025-06-22', '2025-06-23', '2025-06-24', '2025-06-25', '2025-06-26', '2025-10-21', '2026-09-18', '2026-09-19']


See the Monthly File Availability

In [33]:
# Monthly file availability from the S3 manifest only

calendar_df = pd.DataFrame({
    "date": pd.date_range(
        start=START_DATE,
        end=END_DATE,
        freq="D"
    )
})

available_dates = set(file_manifest["date"].dt.normalize())

calendar_df["file_available"] = calendar_df["date"].isin(available_dates)
calendar_df["year_month"] = calendar_df["date"].dt.to_period("M").astype(str)

monthly_coverage = (
    calendar_df
    .groupby("year_month")
    .agg(
        expected_days=("date", "count"),
        available_days=("file_available", "sum")
    )
    .reset_index()
)

monthly_coverage["missing_days"] = (
    monthly_coverage["expected_days"]
    - monthly_coverage["available_days"]
)

monthly_coverage["availability_pct"] = (
    monthly_coverage["available_days"]
    / monthly_coverage["expected_days"]
    * 100
)

# Worst months first
monthly_coverage = monthly_coverage.sort_values(
    ["missing_days", "availability_pct"],
    ascending=[False, True]
).reset_index(drop=True)

monthly_coverage

,year_month,expected_days,available_days,missing_days,availability_pct
0,2025-04,30,0,30,0.000000
1,2025-02,28,0,28,0.000000
2,2025-03,31,3,28,9.677419
3,2025-05,31,9,22,29.032258
4,2025-06,30,13,17,43.333333
5,2025-01,31,14,17,45.161290
6,2024-11,30,27,3,90.000000
7,2026-09,30,28,2,93.333333
8,2024-10,31,29,2,93.548387
9,2024-05,16,15,1,93.750000


In [34]:
monthly_coverage.sort_values(
    "availability_pct",
    ascending=False
).reset_index(drop=True)

,year_month,expected_days,available_days,missing_days,availability_pct
0,2024-08,31,31,0,100.000000
1,2024-07,31,31,0,100.000000
2,2024-06,30,30,0,100.000000
3,2026-04,30,30,0,100.000000
4,2026-03,31,31,0,100.000000
5,2026-02,28,28,0,100.000000
6,2026-01,31,31,0,100.000000
7,2025-12,31,31,0,100.000000
8,2025-11,30,30,0,100.000000
9,2025-09,30,30,0,100.000000


In [35]:
# Monthly coverage in chronological order

monthly_chronological = monthly_coverage.sort_values(
    "year_month"
).reset_index(drop=True)

monthly_chronological

,year_month,expected_days,available_days,missing_days,availability_pct
0,2024-05,16,15,1,93.750000
1,2024-06,30,30,0,100.000000
2,2024-07,31,31,0,100.000000
3,2024-08,31,31,0,100.000000
4,2024-09,30,30,0,100.000000
5,2024-10,31,29,2,93.548387
6,2024-11,30,27,3,90.000000
7,2024-12,31,31,0,100.000000
8,2025-01,31,14,17,45.161290
9,2025-02,28,0,28,0.000000


In [36]:
monthly_chronological[
    [
        "year_month",
        "expected_days",
        "available_days",
        "missing_days",
        "availability_pct"
    ]
]

,year_month,expected_days,available_days,missing_days,availability_pct
0,2024-05,16,15,1,93.750000
1,2024-06,30,30,0,100.000000
2,2024-07,31,31,0,100.000000
3,2024-08,31,31,0,100.000000
4,2024-09,30,30,0,100.000000
5,2024-10,31,29,2,93.548387
6,2024-11,30,27,3,90.000000
7,2024-12,31,31,0,100.000000
8,2025-01,31,14,17,45.161290
9,2025-02,28,0,28,0.000000


Seeing at the Data Availability, Regular Data is Present only From 2025 July, so start downloading from there

In [37]:
# Select the continuous period to download

DOWNLOAD_START = pd.Timestamp("2025-07-01")
DOWNLOAD_END = pd.Timestamp("2026-10-19")

download_manifest = file_manifest[
    (file_manifest["date"] >= DOWNLOAD_START) &
    (file_manifest["date"] <= DOWNLOAD_END)
].copy()

download_manifest = download_manifest.sort_values("date").reset_index(drop=True)

print("Download period:")
print(f"  Start: {download_manifest['date'].min().date()}")
print(f"  End:   {download_manifest['date'].max().date()}")
print(f"  Files: {len(download_manifest)}")

Download period:
  Start: 2025-07-01
  End:   2026-10-19
  Files: 473


In [38]:
print(
    download_manifest[
        ["date", "s3_key"]
    ].head()
)

print("...")
    
print(
    download_manifest[
        ["date", "s3_key"]
    ].tail()
)

        date                                             s3_key
0 2025-07-01  records/csv.gz/locationid=2848864/year=2025/mo...
1 2025-07-02  records/csv.gz/locationid=2848864/year=2025/mo...
2 2025-07-03  records/csv.gz/locationid=2848864/year=2025/mo...
3 2025-07-04  records/csv.gz/locationid=2848864/year=2025/mo...
4 2025-07-05  records/csv.gz/locationid=2848864/year=2025/mo...
...
          date                                             s3_key
468 2026-10-15  records/csv.gz/locationid=2848864/year=2026/mo...
469 2026-10-16  records/csv.gz/locationid=2848864/year=2026/mo...
470 2026-10-17  records/csv.gz/locationid=2848864/year=2026/mo...
471 2026-10-18  records/csv.gz/locationid=2848864/year=2026/mo...
472 2026-10-19  records/csv.gz/locationid=2848864/year=2026/mo...


In [40]:
print(f"\nFiles to download: {len(download_manifest)}")


Files to download: 473


In [41]:
row = download_manifest.iloc[0]

print(row["date"])
print(row["s3_key"])

2025-07-01 00:00:00
records/csv.gz/locationid=2848864/year=2025/month=07/location-2848864-20250701.csv.gz


Try with 1 File first

In [42]:
import requests

test_key = download_manifest.iloc[0]["s3_key"]

url = f"https://openaq-data-archive.s3.amazonaws.com/{test_key}"

r = requests.get(url, stream=True)

print("Status:", r.status_code)
print("Content-Length:", r.headers.get("Content-Length"))
print("Content-Type:", r.headers.get("Content-Type"))

Status: 200
Content-Length: 1938
Content-Type: binary/octet-stream


In [43]:
import requests
from pathlib import Path

row = download_manifest.iloc[0]

file_date = row["date"]
s3_key = row["s3_key"]

url = f"https://openaq-data-archive.s3.amazonaws.com/{s3_key}"
local_file = TEMP_DIR / Path(s3_key).name

print("Downloading:", file_date.date())
print("URL:", url)

r = requests.get(url, timeout=60)

print("Status:", r.status_code)
print("Size:", len(r.content), "bytes")

if r.status_code == 200:
    local_file.write_bytes(r.content)
    print("Saved:", local_file)
else:
    print("ERROR:", r.text[:500])

Downloading: 2025-07-01
URL: https://openaq-data-archive.s3.amazonaws.com/records/csv.gz/locationid=2848864/year=2025/month=07/location-2848864-20250701.csv.gz
Status: 200
Size: 1938 bytes
Saved: Dataset\raw\s3_files_hadigaun_latest\location-2848864-20250701.csv.gz


In [44]:
import pandas as pd

test_df = pd.read_csv(
    local_file,
    compression="gzip"
)

print(test_df.shape)
test_df.head()

(120, 9)


,location_id,sensors_id,location,datetime,lat,lon,parameter,units,value
0,2848864,9235746,"Dabali, Handigaun-2818823",2025-07-01T00:45:00+05:45,27.681891,85.287078,pm25,µg/m³,13.717708
1,2848864,9235746,"Dabali, Handigaun-2818823",2025-07-01T01:45:00+05:45,27.681891,85.287078,pm25,µg/m³,11.052827
2,2848864,9235746,"Dabali, Handigaun-2818823",2025-07-01T02:45:00+05:45,27.681891,85.287078,pm25,µg/m³,9.482589
3,2848864,9235746,"Dabali, Handigaun-2818823",2025-07-01T03:45:00+05:45,27.681891,85.287078,pm25,µg/m³,10.492857
4,2848864,9235746,"Dabali, Handigaun-2818823",2025-07-01T04:45:00+05:45,27.681891,85.287078,pm25,µg/m³,23.859673


Now, download all the files

In [45]:
import requests
from pathlib import Path

download_log = []

for i, row in download_manifest.iterrows():

    file_date = row["date"]
    s3_key = row["s3_key"]

    local_file = TEMP_DIR / Path(s3_key).name
    url = f"https://openaq-data-archive.s3.amazonaws.com/{s3_key}"

    if local_file.exists() and local_file.stat().st_size > 0:

        status = "skipped"
        error = ""

        print(
            f"[{i+1}/{len(download_manifest)}] "
            f"{file_date.date()} — exists"
        )

    else:

        print(
            f"[{i+1}/{len(download_manifest)}] "
            f"{file_date.date()} — downloading"
        )

        try:
            r = requests.get(
                url,
                timeout=120
            )

            if r.status_code == 200:

                local_file.write_bytes(r.content)

                status = "downloaded"
                error = ""

            else:

                status = "failed"
                error = f"HTTP {r.status_code}: {r.text[:500]}"

                print("  ERROR:", error)

        except Exception as e:

            status = "failed"
            error = str(e)

            print("  ERROR:", error)

    download_log.append({
        "date": file_date,
        "s3_key": s3_key,
        "local_file": str(local_file),
        "status": status,
        "size_bytes": (
            local_file.stat().st_size
            if local_file.exists()
            else None
        ),
        "error": error
    })

download_log_df = pd.DataFrame(download_log)

print("\nDownload summary:")
print(download_log_df["status"].value_counts())

[1/473] 2025-07-01 — exists
[2/473] 2025-07-02 — exists
[3/473] 2025-07-03 — exists
[4/473] 2025-07-04 — exists
[5/473] 2025-07-05 — exists
[6/473] 2025-07-06 — exists
[7/473] 2025-07-07 — exists
[8/473] 2025-07-08 — exists
[9/473] 2025-07-09 — exists
[10/473] 2025-07-10 — exists
[11/473] 2025-07-11 — exists
[12/473] 2025-07-12 — exists
[13/473] 2025-07-13 — exists
[14/473] 2025-07-14 — exists
[15/473] 2025-07-15 — exists
[16/473] 2025-07-16 — exists
[17/473] 2025-07-17 — exists
[18/473] 2025-07-18 — exists
[19/473] 2025-07-19 — exists
[20/473] 2025-07-20 — exists
[21/473] 2025-07-21 — exists
[22/473] 2025-07-22 — exists
[23/473] 2025-07-23 — exists
[24/473] 2025-07-24 — exists
[25/473] 2025-07-25 — exists
[26/473] 2025-07-26 — exists
[27/473] 2025-07-27 — exists
[28/473] 2025-07-28 — exists
[29/473] 2025-07-29 — exists
[30/473] 2025-07-30 — exists
[31/473] 2025-07-31 — exists
[32/473] 2025-08-01 — exists
[33/473] 2025-08-02 — exists
[34/473] 2025-08-03 — exists
[35/473] 2025-08-04 — e

Check the validity of Files

In [46]:
import gzip

valid_files = []
invalid_files = []

for file_path in TEMP_DIR.glob("*.csv.gz"):

    try:
        with gzip.open(file_path, "rb") as f:
            f.read(10)

        valid_files.append(file_path)

    except Exception:
        invalid_files.append(file_path)

print("Valid gzip files:", len(valid_files))
print("Invalid files:", len(invalid_files))

Valid gzip files: 473
Invalid files: 0


In [47]:
# Inspect the structure of all downloaded raw files

file_inventory = []

for i, file_path in enumerate(sorted(TEMP_DIR.glob("*.csv.gz")), start=1):

    df = pd.read_csv(
        file_path,
        compression="gzip"
    )

    file_inventory.append({
        "file": file_path.name,
        "date": file_path.name[-15:-7],  # YYYYMMDD
        "rows": len(df),
        "columns": len(df.columns),
        "parameters": ", ".join(
            sorted(df["parameter"].dropna().unique())
        ),
        "min_datetime": df["datetime"].min(),
        "max_datetime": df["datetime"].max()
    })

    if i % 50 == 0 or i == 454:
        print(f"Inspected {i}/454 files")

file_inventory_df = pd.DataFrame(file_inventory)

file_inventory_df.head()

Inspected 50/454 files
Inspected 100/454 files
Inspected 150/454 files
Inspected 200/454 files
Inspected 250/454 files
Inspected 300/454 files
Inspected 350/454 files
Inspected 400/454 files
Inspected 450/454 files
Inspected 454/454 files


,file,date,rows,columns,parameters,min_datetime,max_datetime
0,location-2848864-20250701.csv.gz,20250701,120,9,"pm1, pm25, relativehumidity, temperature, um003",2025-07-01T00:45:00+05:45,2025-07-01T23:45:00+05:45
1,location-2848864-20250702.csv.gz,20250702,120,9,"pm1, pm25, relativehumidity, temperature, um003",2025-07-02T00:45:00+05:45,2025-07-02T23:45:00+05:45
2,location-2848864-20250703.csv.gz,20250703,120,9,"pm1, pm25, relativehumidity, temperature, um003",2025-07-03T00:45:00+05:45,2025-07-03T23:45:00+05:45
3,location-2848864-20250704.csv.gz,20250704,120,9,"pm1, pm25, relativehumidity, temperature, um003",2025-07-04T00:45:00+05:45,2025-07-04T23:45:00+05:45
4,location-2848864-20250705.csv.gz,20250705,120,9,"pm1, pm25, relativehumidity, temperature, um003",2025-07-05T00:45:00+05:45,2025-07-05T23:45:00+05:45


In [48]:
print("Files inspected:", len(file_inventory_df))
print("Total raw rows:", file_inventory_df["rows"].sum())

file_inventory_df.describe(include="all")

Files inspected: 473
Total raw rows: 54905


,file,date,rows,columns,parameters,min_datetime,max_datetime
count,473,473,473.000000,473.0,473,473,473
unique,473,473,NaN,NaN,1,473,473
top,location-2848864-20250701.csv.gz,20250701,NaN,NaN,"pm1, pm25, relativehumidity, temperature, um003",2025-07-01T00:45:00+05:45,2025-07-01T23:45:00+05:45
freq,1,1,NaN,NaN,473,1,1
mean,NaN,NaN,116.078224,9.0,NaN,NaN,NaN
std,NaN,NaN,14.368789,0.0,NaN,NaN,NaN
min,NaN,NaN,5.000000,9.0,NaN,NaN,NaN
25%,NaN,NaN,120.000000,9.0,NaN,NaN,NaN
50%,NaN,NaN,120.000000,9.0,NaN,NaN,NaN
75%,NaN,NaN,120.000000,9.0,NaN,NaN,NaN


In [49]:
parameter_counts = {}

for file_path in sorted(TEMP_DIR.glob("*.csv.gz")):

    df = pd.read_csv(
        file_path,
        compression="gzip",
        usecols=["parameter"]
    )

    for parameter in df["parameter"].dropna().unique():
        parameter_counts[parameter] = (
            parameter_counts.get(parameter, 0) + 1
        )

parameter_counts

{'pm25': 473,
 'pm1': 473,
 'relativehumidity': 473,
 'temperature': 473,
 'um003': 473}

Combine all downloaded raw files into a single CSV

In [50]:
# Combine all downloaded raw files into one CSV

dataframes = []

raw_files = sorted(TEMP_DIR.glob("*.csv.gz"))

print(f"Files to combine: {len(raw_files)}")

for i, file_path in enumerate(raw_files, start=1):

    df = pd.read_csv(
        file_path,
        compression="gzip"
    )

    dataframes.append(df)

    if i % 50 == 0 or i == len(raw_files):
        print(f"Read {i}/{len(raw_files)} files")

raw_df = pd.concat(
    dataframes,
    ignore_index=True
)

print("\nCombined dataset shape:")
print(raw_df.shape)

Files to combine: 473
Read 50/473 files
Read 100/473 files
Read 150/473 files
Read 200/473 files
Read 250/473 files
Read 300/473 files
Read 350/473 files
Read 400/473 files
Read 450/473 files
Read 473/473 files

Combined dataset shape:
(54905, 9)


Convert the file to CSV

In [51]:
raw_df.to_csv(
    raw_csv := OUTPUT_FILE,
    index=False
)

print(f"Saved: {raw_csv}")
print(f"Rows: {len(raw_df):,}")

Saved: Dataset\raw_data_hadigaun_latest.csv
Rows: 54,905


In [52]:
print(raw_df.columns.tolist())
print()
print(raw_df["parameter"].value_counts())

['location_id', 'sensors_id', 'location', 'datetime', 'lat', 'lon', 'parameter', 'units', 'value']

parameter
pm25                10981
pm1                 10981
relativehumidity    10981
temperature         10981
um003               10981
Name: count, dtype: int64


Similarly, using the same Methods to download raw data for Pokhara (Location_id = 6088772, Timeline 2025 Nov - 2026 Sept), Lalitpur (Location_id = 6098754, Timeline 2025 Nov - 2026 Sept), Embassy (Location_id = 3459, Timeline 2022 Oct - 2025 Feb), and Durbar (Location_id = 3460, Timeline 2022 Oct - 2025 Feb)

## AQI Data Preprocessing

Preprocess the raw CSV Data taken from the API, and convert it into required format for further Processing

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
DATASET_DIR = Path(r'D:/MSDSA/4th Sem/Thesis/Dataset')

Read the raw data from the CSV file

In [ ]:
# read the raw data CSV file
RAW_CSV = DATASET_DIR / "raw_data_hadigaun.csv"
raw_df = pd.read_csv(RAW_CSV)

In [ ]:
raw_df.head()

,location_id,sensors_id,location,datetime,lat,lon,parameter,units,value
0,2848864,9235746,"Dabali, Handigaun-2818823",2025-07-01T00:45:00+05:45,27.681891,85.287078,pm25,µg/m³,13.717708
1,2848864,9235746,"Dabali, Handigaun-2818823",2025-07-01T01:45:00+05:45,27.681891,85.287078,pm25,µg/m³,11.052827
2,2848864,9235746,"Dabali, Handigaun-2818823",2025-07-01T02:45:00+05:45,27.681891,85.287078,pm25,µg/m³,9.482589
3,2848864,9235746,"Dabali, Handigaun-2818823",2025-07-01T03:45:00+05:45,27.681891,85.287078,pm25,µg/m³,10.492857
4,2848864,9235746,"Dabali, Handigaun-2818823",2025-07-01T04:45:00+05:45,27.681891,85.287078,pm25,µg/m³,23.859673


In [ ]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 54905 entries, 0 to 54904
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   location_id  54905 non-null  int64  
 1   sensors_id   54905 non-null  int64  
 2   location     54905 non-null  str    
 3   datetime     54905 non-null  str    
 4   lat          54905 non-null  float64
 5   lon          54905 non-null  float64
 6   parameter    54905 non-null  str    
 7   units        54905 non-null  str    
 8   value        54905 non-null  float64
dtypes: float64(3), int64(2), str(4)
memory usage: 3.8 MB


In [ ]:
raw_df["parameter"].value_counts()

parameter
pm25                10981
pm1                 10981
relativehumidity    10981
temperature         10981
um003               10981
Name: count, dtype: int64

Check for invalid datetime

In [ ]:
raw_df["datetime"] = pd.to_datetime(
    raw_df["datetime"],
    errors="coerce"
)

print("Invalid datetime values:", raw_df["datetime"].isna().sum())

Invalid datetime values: 0


Check for Nulls, missing values, or duplicates

In [ ]:
raw_df.isna().sum()

location_id    0
sensors_id     0
location       0
datetime       0
lat            0
lon            0
parameter      0
units          0
value          0
dtype: int64

In [ ]:
raw_df.groupby(
    ["parameter", "units"]
).size().reset_index(name="count")

,parameter,units,count
0,pm1,µg/m³,10981
1,pm25,µg/m³,10981
2,relativehumidity,%,10981
3,temperature,c,10981
4,um003,particles/cm³,10981


In [ ]:
print("Start:", raw_df["datetime"].min())
print("End:", raw_df["datetime"].max())

Start: 2025-07-01 00:45:00+05:45
End: 2026-10-19 08:45:00+05:45


In [ ]:
parameters = sorted(
    raw_df["parameter"].dropna().unique()
)

print(parameters)

['pm1', 'pm25', 'relativehumidity', 'temperature', 'um003']


In [ ]:
duplicate_count = raw_df.duplicated(
    subset=["datetime", "parameter"]
).sum()

print("Duplicate datetime-parameter rows:", duplicate_count)

Duplicate datetime-parameter rows: 0


Pivot the dataframe

In [ ]:
wide_df = (
    raw_df
    .pivot_table(
        index="datetime",
        columns="parameter",
        values="value",
        aggfunc="mean"
    )
    .reset_index()
)

In [ ]:
wide_df.columns.name = None

wide_df = wide_df[
    [
        "datetime",
        "pm25",
        "pm1",
        "temperature",
        "relativehumidity",
        "um003"
    ]
]

In [ ]:
print("Shape:", wide_df.shape)

wide_df.head()

Shape: (10981, 6)


,datetime,pm25,pm1,temperature,relativehumidity,um003
0,2025-07-01 00:45:00+05:45,13.717708,9.164286,24.237649,64.782738,1849.218770
1,2025-07-01 01:45:00+05:45,11.052827,7.255308,23.981399,65.276935,1480.463806
2,2025-07-01 02:45:00+05:45,9.482589,6.263244,23.723066,66.015773,1320.891368
3,2025-07-01 03:45:00+05:45,10.492857,6.811310,23.914285,65.384673,1410.139893
4,2025-07-01 04:45:00+05:45,23.859673,16.167559,23.752083,65.747619,3319.800588


In [ ]:
wide_df.isna().sum()

datetime            0
pm25                0
pm1                 0
temperature         0
relativehumidity    0
um003               0
dtype: int64

In [ ]:
missing_pct = (
    wide_df.isna().mean() * 100
).sort_values(ascending=False)

missing_pct

datetime            0.0
pm25                0.0
pm1                 0.0
temperature         0.0
relativehumidity    0.0
um003               0.0
dtype: float64

In [ ]:
time_diff = (
    wide_df["datetime"]
    .sort_values()
    .diff()
)

time_diff.value_counts().head(20)

datetime
0 days 01:00:00    10896
0 days 02:00:00       47
0 days 04:00:00        8
0 days 03:00:00        7
0 days 05:00:00        5
0 days 06:00:00        4
0 days 07:00:00        4
1 days 20:00:00        2
0 days 21:00:00        1
0 days 17:00:00        1
1 days 21:00:00        1
0 days 09:00:00        1
0 days 16:00:00        1
0 days 10:00:00        1
3 days 10:00:00        1
Name: count, dtype: int64

In [ ]:
wide_df["date"] = wide_df["datetime"].dt.date

daily_counts = (
    wide_df
    .groupby("date")
    .size()
)

daily_counts.describe()

count    473.000000
mean      23.215645
std        2.873758
min        1.000000
25%       24.000000
50%       24.000000
75%       24.000000
max       24.000000
dtype: float64

Check for missing observations within the days

In [ ]:
print("Minimum observations/day:", daily_counts.min())
print("Maximum observations/day:", daily_counts.max())

Minimum observations/day: 1
Maximum observations/day: 24


In [ ]:
daily_counts.sort_values().head(25)

date
2026-05-09     1
2026-06-11     2
2026-06-10     3
2026-05-08     4
2025-08-20     5
2026-09-20     6
2026-09-17     9
2026-10-19     9
2026-04-28    11
2025-10-20    11
2025-09-09    15
2026-07-03    15
2025-11-14    15
2025-09-10    17
2025-10-22    17
2026-07-22    18
2025-11-12    18
2025-09-15    18
2026-08-19    18
2026-06-06    18
2026-09-27    18
2026-06-07    19
2025-09-11    19
2025-09-08    19
2026-09-26    19
dtype: int64

In [ ]:
expected_hourly = pd.date_range(
    start=wide_df["datetime"].min(),
    end=wide_df["datetime"].max(),
    freq="1h",
    tz=wide_df["datetime"].dt.tz
)

actual_timestamps = pd.DatetimeIndex(wide_df["datetime"])

missing_timestamps = expected_hourly.difference(actual_timestamps)

print("Expected timestamps:", len(expected_hourly))
print("Actual timestamps:", len(actual_timestamps))
print("Missing timestamps:", len(missing_timestamps))

Expected timestamps: 11409
Actual timestamps: 10981
Missing timestamps: 428


In [ ]:
missing_timestamps[:20]

DatetimeIndex(['2025-07-09 13:45:00+05:45', '2025-07-18 08:45:00+05:45',
               '2025-08-19 23:45:00+05:45', '2025-08-20 00:45:00+05:45',
               '2025-08-20 01:45:00+05:45', '2025-08-20 02:45:00+05:45',
               '2025-08-20 03:45:00+05:45', '2025-08-20 04:45:00+05:45',
               '2025-08-20 05:45:00+05:45', '2025-08-20 06:45:00+05:45',
               '2025-08-20 07:45:00+05:45', '2025-08-20 08:45:00+05:45',
               '2025-08-20 09:45:00+05:45', '2025-08-20 10:45:00+05:45',
               '2025-08-20 11:45:00+05:45', '2025-08-20 12:45:00+05:45',
               '2025-08-20 13:45:00+05:45', '2025-08-20 14:45:00+05:45',
               '2025-08-20 15:45:00+05:45', '2025-08-20 16:45:00+05:45'],
              dtype='datetime64[us, UTC+05:45]', freq=None)

In [ ]:
print("Missing percentage:",
      len(missing_timestamps) / len(expected_hourly) * 100)

Missing percentage: 3.751424314137961


In [ ]:
wide_df.head()

,datetime,pm25,pm1,temperature,relativehumidity,um003,date
0,2025-07-01 00:45:00+05:45,13.717708,9.164286,24.237649,64.782738,1849.218770,2025-07-01
1,2025-07-01 01:45:00+05:45,11.052827,7.255308,23.981399,65.276935,1480.463806,2025-07-01
2,2025-07-01 02:45:00+05:45,9.482589,6.263244,23.723066,66.015773,1320.891368,2025-07-01
3,2025-07-01 03:45:00+05:45,10.492857,6.811310,23.914285,65.384673,1410.139893,2025-07-01
4,2025-07-01 04:45:00+05:45,23.859673,16.167559,23.752083,65.747619,3319.800588,2025-07-01


In [ ]:
STANDARD_CSV = DATASET_DIR / "air_quality_data_hadigaun.csv"

wide_df.to_csv(
    STANDARD_CSV,
    index=False
)

print(f"Saved: {STANDARD_CSV}")
print("Columns:", wide_df.columns.tolist())

Saved: D:\MSDSA\4th Sem\Thesis\Dataset\air_quality_data_hadigaun.csv
Columns: ['datetime', 'pm25', 'pm1', 'temperature', 'relativehumidity', 'um003', 'date']


Similarly for Datasets of Other Location. The raw data is present in the same folder.

## Weather data extraction (Open-Meteo, Hadigaun)
Weather variables are requested from the Open-Meteo Historical Weather API for the Hadigaun station
coordinates. Settings: model `ecmwf_ifs`, grid cell `nearest` (4.0 km from the target, chosen after
comparing ecmwf_ifs, era5 and era5_land: era5_land lacks most variables, era5 is 8.4 km away on a 0.25° grid),
default elevation handling (reference height 1325 m from the 90 m elevation model), and UTC timestamps, so values
are hourly on the hour in UTC. The request starts on 2025-06-30 (UTC) so that 1 July 2025 is complete in Nepal time.

In [ ]:
import json, requests, pandas as pd
from pathlib import Path
from datetime import datetime, timezone

LAT, LON = 27.68189114, 85.28707804        # Hadigaun station
START, END = "2025-06-30", "2026-10-31"    # UTC dates
OUT = Path("data/raw"); OUT.mkdir(parents=True, exist_ok=True)

The 20 candidate variables are requested in one call so they share the same grid cell, model run and download time.

In [ ]:
weather_vars = [
    "temperature_2m", "relative_humidity_2m", "dew_point_2m", "precipitation", "rain",
    "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m", "surface_pressure", "pressure_msl",
    "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high", "boundary_layer_height",
    "shortwave_radiation", "direct_radiation", "diffuse_radiation", "terrestrial_radiation", "sunshine_duration",
]

The Request

In [ ]:
url = "https://archive-api.open-meteo.com/v1/archive"
params = {"latitude": LAT, "longitude": LON, "start_date": START, "end_date": END,
          "hourly": ",".join(weather_vars), "models": "ecmwf_ifs", "cell_selection": "nearest",
          "timezone": "GMT", "timeformat": "unixtime", "wind_speed_unit": "ms", "precipitation_unit": "mm"}
resp = requests.get(url, params=params, timeout=180)
resp.raise_for_status()
meta = resp.json()

The response is saved exactly as returned, together with the request URL, so the download can be reproduced and audited.

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
(OUT / f"openmeteo_weather_{stamp}.json").write_text(resp.text)
(OUT / f"openmeteo_weather_{stamp}_request.txt").write_text(resp.url)
print(stamp, "|", meta["latitude"], meta["longitude"], "| elevation", meta["elevation"], "m")

20261112T090446Z | 27.662565 85.252525 | elevation 1325.0 m


Conversion into Table

In [ ]:
wx = pd.DataFrame(meta["hourly"])
wx["datetime_utc"] = pd.to_datetime(wx.pop("time"), unit="s", utc=True)
wx = wx[["datetime_utc"] + weather_vars]
print(wx.shape, wx["datetime_utc"].min(), wx["datetime_utc"].max())

(11736, 21) 2025-06-30 00:00:00+00:00 2026-10-31 23:00:00+00:00


Integrity Checks

In [ ]:
print("Step sizes:", wx["datetime_utc"].diff().value_counts().to_dict())
print("Duplicate timestamps:", wx["datetime_utc"].duplicated().sum())
nulls = wx.isna().sum()
print(nulls[nulls > 0])

Step sizes: {Timedelta('0 days 01:00:00'): 11735}
Duplicate timestamps: 0
boundary_layer_height    492
dtype: int64


Save the Raw CSV and Units

In [ ]:
wx.to_csv(OUT / "Hadigaun_weather.csv", index=False)
pd.Series(meta["hourly_units"], name="unit").to_csv(OUT / "Hadigaun_weather_units.csv")

## Modelled air-quality data (Open-Meteo Air Quality API, CAMS global)
Modelled PM2.5, PM10, NO2, SO2, CO and aerosol optical depth are requested for the Hadigaun coordinates from the
Open-Meteo Air Quality API (CAMS global domain). These are atmospheric-model values on a coarse grid, not
measurements, and are used only for comparison with the sensor data. The native step is 3-hourly, so hourly values are
interpolated. The request starts at 2022-08-01, the earliest date expected to be available; the first valid timestamp
of each variable is checked below. Attribution: CAMS / Open-Meteo.

In [ ]:
aq_url = "https://air-quality-api.open-meteo.com/v1/air-quality"
aq_vars = ["pm2_5", "pm10", "nitrogen_dioxide", "sulphur_dioxide", "carbon_monoxide", "aerosol_optical_depth"]
aq_params = {"latitude": LAT, "longitude": LON, "start_date": "2022-08-01", "end_date": END,
             "hourly": ",".join(aq_vars), "domains": "cams_global", "timezone": "GMT", "timeformat": "unixtime"}

Request

In [ ]:
aq_resp = requests.get(aq_url, params=aq_params, timeout=180)
aq_resp.raise_for_status()
aq_meta = aq_resp.json()
print(aq_meta["latitude"], aq_meta["longitude"], aq_meta.get("elevation"))

27.700005 85.30002 1325.0


The cell returned for the air-quality model is checked against the target coordinates, and the raw response is saved untouched.

In [ ]:
import numpy as np
dist_km = np.hypot((aq_meta["latitude"] - LAT) * 111.2, (aq_meta["longitude"] - LON) * 111.2 * np.cos(np.radians(LAT)))
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
(OUT / f"openmeteo_airquality_{stamp}.json").write_text(aq_resp.text, encoding="utf-8")
(OUT / f"openmeteo_airquality_{stamp}_request.txt").write_text(aq_resp.url, encoding="utf-8")
print(f"Cell is about {dist_km:.1f} km from the target | saved {stamp}")

Cell is about 2.4 km from the target | saved 20261112T090957Z


Converting to Tables

In [ ]:
aq = pd.DataFrame(aq_meta["hourly"])
aq["datetime_utc"] = pd.to_datetime(aq.pop("time"), unit="s", utc=True)
aq = aq[["datetime_utc"] + aq_vars]
print(aq.shape, aq["datetime_utc"].min(), aq["datetime_utc"].max())

(37272, 7) 2022-08-01 00:00:00+00:00 2026-10-31 23:00:00+00:00


In [ ]:
print("Step sizes:", aq["datetime_utc"].diff().value_counts().to_dict())
print("Duplicates:", aq["datetime_utc"].duplicated().sum())
print("First valid timestamp:\n", aq.set_index("datetime_utc").apply(pd.Series.first_valid_index))
print("Nulls:\n", aq.isna().sum())

Step sizes: {Timedelta('0 days 01:00:00'): 37271}
Duplicates: 0
First valid timestamp:
 pm2_5                   2022-08-04 00:00:00+00:00
pm10                    2022-08-04 00:00:00+00:00
nitrogen_dioxide        2022-08-04 00:00:00+00:00
sulphur_dioxide         2022-08-04 00:00:00+00:00
carbon_monoxide         2022-08-04 00:00:00+00:00
aerosol_optical_depth   2022-08-04 00:00:00+00:00
dtype: datetime64[s, UTC]
Nulls:
 datetime_utc              0
pm2_5                    72
pm10                     72
nitrogen_dioxide         72
sulphur_dioxide          72
carbon_monoxide          72
aerosol_optical_depth    72
dtype: int64


Save it to CSV

In [ ]:
aq.to_csv(OUT / "Hadigaun_airquality_cams_raw.csv", index=False)
pd.Series(aq_meta["hourly_units"], name="unit").to_csv(OUT / "Hadigaun_airquality_units.csv")

## Master Dataset Creation
Now, merging the AQI Data and the Weather Data of Hadigaun to create a master dataset used for further Analysis. The AQI Dataset is taken as the base, and subsequent weather data for the timestamp present in the AQI Dataset is added.

In [ ]:
# Dataset directory (one level above Codes)
DATASET_DIR = Path("../Dataset")

# Load CSV files
air_quality_df = pd.read_csv(
    DATASET_DIR / "air_quality_data_hadigaun.csv"
)

weather_df = pd.read_csv(
    DATASET_DIR / "weather_data_hadigaun.csv"
)

print("Air Quality Dataset Shape:", air_quality_df.shape)
print("Weather Dataset Shape:", weather_df.shape)

Air Quality Dataset Shape: (10540, 6)
Weather Dataset Shape: (10968, 15)


In [ ]:
# ==========================================
# AIR QUALITY DATASET
# ==========================================

print("AIR QUALITY DATASET")
print("=" * 60)

print("\nColumns:")
print(air_quality_df.columns.tolist())

print("\nData Types:")
print(air_quality_df.dtypes)

print("\nFirst 5 Records:")
display(air_quality_df.head())


# ==========================================
# WEATHER DATASET
# ==========================================

print("\nWEATHER DATASET")
print("=" * 60)

print("\nColumns:")
print(weather_df.columns.tolist())

print("\nData Types:")
print(weather_df.dtypes)

print("\nFirst 5 Records:")
display(weather_df.head())

AIR QUALITY DATASET

Columns:
['Datetime', 'PM2.5', 'PM1', 'Temperature', 'Relative_Humidity', 'Particle_Count_0.3um']

Data Types:
Datetime                    str
PM2.5                   float64
PM1                     float64
Temperature             float64
Relative_Humidity       float64
Particle_Count_0.3um    float64
dtype: object

First 5 Records:


,Datetime,PM2.5,PM1,Temperature,Relative_Humidity,Particle_Count_0.3um
0,2025-07-01 00:45:00+05:45,13.717708,9.164286,24.237649,64.782738,1849.218770
1,2025-07-01 01:45:00+05:45,11.052827,7.255308,23.981399,65.276935,1480.463806
2,2025-07-01 02:45:00+05:45,9.482589,6.263244,23.723066,66.015773,1320.891368
3,2025-07-01 03:45:00+05:45,10.492857,6.811310,23.914285,65.384673,1410.139893
4,2025-07-01 04:45:00+05:45,23.859673,16.167559,23.752083,65.747619,3319.800588



WEATHER DATASET

Columns:
['datetime_local', 'datetime_utc', 'precipitation', 'rain', 'wind_speed_10m', 'wind_direction_10m', 'surface_pressure', 'boundary_layer_height', 'shortwave_radiation', 'direct_radiation', 'diffuse_radiation', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high']

Data Types:
datetime_local               str
datetime_utc                 str
precipitation            float64
rain                     float64
wind_speed_10m           float64
wind_direction_10m         int64
surface_pressure         float64
boundary_layer_height    float64
shortwave_radiation      float64
direct_radiation         float64
diffuse_radiation        float64
cloud_cover                int64
cloud_cover_low            int64
cloud_cover_mid            int64
cloud_cover_high           int64
dtype: object

First 5 Records:


,datetime_local,datetime_utc,precipitation,rain,wind_speed_10m,wind_direction_10m,surface_pressure,boundary_layer_height,shortwave_radiation,direct_radiation,diffuse_radiation,cloud_cover,cloud_cover_low,cloud_cover_mid,cloud_cover_high
0,2025-07-01 00:00:00+05:45,2025-06-30 18:15:00+00:00,0.4,0.4,1.00,174,861.8,45.0,0.0,0.0,0.0,100,1,98,100
1,2025-07-01 01:00:00+05:45,2025-06-30 19:15:00+00:00,0.2,0.2,1.32,169,861.8,55.0,0.0,0.0,0.0,100,40,100,100
2,2025-07-01 02:00:00+05:45,2025-06-30 20:15:00+00:00,0.5,0.5,0.76,169,861.2,35.0,0.0,0.0,0.0,94,4,84,65
3,2025-07-01 03:00:00+05:45,2025-06-30 21:15:00+00:00,0.0,0.0,0.96,171,860.8,30.0,0.0,0.0,0.0,98,5,44,97
4,2025-07-01 04:00:00+05:45,2025-06-30 22:15:00+00:00,0.0,0.0,0.90,180,860.8,25.0,0.0,0.0,0.0,96,3,19,94


In [ ]:
# ==========================================
# TIMESTAMP STANDARDIZATION
# ==========================================

TIMEZONE = "Asia/Kathmandu"

# Preserve original timestamps
air_quality_df["datetime_openaq_original"] = air_quality_df["Datetime"]

weather_df["datetime_weather_original"] = weather_df["datetime_local"]

# Parse OpenAQ timestamps
air_quality_df["datetime_openaq"] = (
    pd.to_datetime(
        air_quality_df["Datetime"],
        utc=True
    ).dt.tz_convert(TIMEZONE)
)

# Parse Open-Meteo timestamps
weather_df["datetime_weather"] = (
    pd.to_datetime(
        weather_df["datetime_local"],
        utc=True
    ).dt.tz_convert(TIMEZONE)
)

# Create standardized hourly join keys
air_quality_df["datetime_hour"] = (
    air_quality_df["datetime_openaq"].dt.floor("h")
)

weather_df["datetime_hour"] = (
    weather_df["datetime_weather"].dt.floor("h")
)

print("Timestamp standardization completed.")

Timestamp standardization completed.


In [ ]:
# ==========================================
# TIMESTAMP ALIGNMENT ANALYSIS
# ==========================================

print("AIR QUALITY TIMESTAMP DISTRIBUTION")
print("=" * 50)

print("\nMinute values:")
display(
    air_quality_df["datetime_openaq"]
    .dt.minute.value_counts()
    .sort_index()
)

print("\nSecond values:")
display(
    air_quality_df["datetime_openaq"]
    .dt.second.value_counts()
    .sort_index()
)

print("\nWEATHER TIMESTAMP DISTRIBUTION")
print("=" * 50)

print("\nMinute values:")
display(
    weather_df["datetime_weather"]
    .dt.minute.value_counts()
    .sort_index()
)

# Check duplicates after hourly standardization
print("\nDuplicate hourly keys")

print(
    "OpenAQ:",
    air_quality_df["datetime_hour"].duplicated().sum()
)

print(
    "Open-Meteo:",
    weather_df["datetime_hour"].duplicated().sum()
)

# Check date ranges
print("\nDATE RANGES")

print(
    "OpenAQ:",
    air_quality_df["datetime_openaq"].min(),
    "to",
    air_quality_df["datetime_openaq"].max()
)

print(
    "Weather:",
    weather_df["datetime_weather"].min(),
    "to",
    weather_df["datetime_weather"].max()
)

AIR QUALITY TIMESTAMP DISTRIBUTION

Minute values:


datetime_openaq
45    10540
Name: count, dtype: int64


Second values:


datetime_openaq
0    10540
Name: count, dtype: int64


WEATHER TIMESTAMP DISTRIBUTION

Minute values:


datetime_weather
0    10968
Name: count, dtype: int64


Duplicate hourly keys
OpenAQ: 0
Open-Meteo: 0

DATE RANGES
OpenAQ: 2025-07-01 00:45:00+05:45 to 2026-09-30 23:45:00+05:45
Weather: 2025-07-01 00:00:00+05:45 to 2026-09-30 23:00:00+05:45


In [ ]:
# ==========================================
# PREPARE DATASETS FOR MERGING
# ==========================================

# OpenAQ: preserve original timestamp and five variables
aq_columns = [
    "datetime_hour",
    "datetime_openaq_original",
    "PM2.5",
    "PM1",
    "Temperature",
    "Relative_Humidity",
    "Particle_Count_0.3um"
]

aq_merge_df = air_quality_df[aq_columns].copy()


# Open-Meteo: preserve original timestamps and 13 variables
weather_columns = [
    "datetime_hour",
    "datetime_weather_original",
    "datetime_utc",
    "precipitation",
    "rain",
    "wind_speed_10m",
    "wind_direction_10m",
    "surface_pressure",
    "boundary_layer_height",
    "shortwave_radiation",
    "direct_radiation",
    "diffuse_radiation",
    "cloud_cover",
    "cloud_cover_low",
    "cloud_cover_mid",
    "cloud_cover_high"
]

weather_merge_df = weather_df[weather_columns].copy()


# ==========================================
# CHECK DUPLICATE JOIN KEYS
# ==========================================

if aq_merge_df["datetime_hour"].duplicated().any():
    raise ValueError(
        "Duplicate hourly timestamps found in OpenAQ."
    )

if weather_merge_df["datetime_hour"].duplicated().any():
    raise ValueError(
        "Duplicate hourly timestamps found in Open-Meteo."
    )


# ==========================================
# MERGE DATASETS
# ==========================================

master_df = pd.merge(
    aq_merge_df,
    weather_merge_df,
    on="datetime_hour",
    how="left",
    validate="one_to_one",
    indicator=True
)

# Rename standardized timestamp
master_df.rename(
    columns={"datetime_hour": "Datetime"},
    inplace=True
)

# Display merge statistics
print("MERGE COMPLETED")
print("=" * 50)

print("Air Quality Records:", len(aq_merge_df))
print("Weather Records:", len(weather_merge_df))
print("Master Dataset Records:", len(master_df))

print("\nMerge Status:")
display(
    master_df["_merge"].value_counts()
)

# Remove technical merge indicator
master_df.drop(columns=["_merge"], inplace=True)

# Sort chronologically
master_df.sort_values(
    "Datetime",
    inplace=True
)

master_df.reset_index(drop=True, inplace=True)

display(master_df.head())

MERGE COMPLETED
Air Quality Records: 10540
Weather Records: 10968
Master Dataset Records: 10540

Merge Status:


_merge
both          10540
left_only         0
right_only        0
Name: count, dtype: int64

,Datetime,datetime_openaq_original,PM2.5,PM1,Temperature,Relative_Humidity,Particle_Count_0.3um,datetime_weather_original,datetime_utc,precipitation,...,wind_direction_10m,surface_pressure,boundary_layer_height,shortwave_radiation,direct_radiation,diffuse_radiation,cloud_cover,cloud_cover_low,cloud_cover_mid,cloud_cover_high
0,2025-07-01 00:00:00+05:45,2025-07-01 00:45:00+05:45,13.717708,9.164286,24.237649,64.782738,1849.218770,2025-07-01 00:00:00+05:45,2025-06-30 18:15:00+00:00,0.4,...,174,861.8,45.0,0.0,0.0,0.0,100,1,98,100
1,2025-07-01 01:00:00+05:45,2025-07-01 01:45:00+05:45,11.052827,7.255308,23.981399,65.276935,1480.463806,2025-07-01 01:00:00+05:45,2025-06-30 19:15:00+00:00,0.2,...,169,861.8,55.0,0.0,0.0,0.0,100,40,100,100
2,2025-07-01 02:00:00+05:45,2025-07-01 02:45:00+05:45,9.482589,6.263244,23.723066,66.015773,1320.891368,2025-07-01 02:00:00+05:45,2025-06-30 20:15:00+00:00,0.5,...,169,861.2,35.0,0.0,0.0,0.0,94,4,84,65
3,2025-07-01 03:00:00+05:45,2025-07-01 03:45:00+05:45,10.492857,6.811310,23.914285,65.384673,1410.139893,2025-07-01 03:00:00+05:45,2025-06-30 21:15:00+00:00,0.0,...,171,860.8,30.0,0.0,0.0,0.0,98,5,44,97
4,2025-07-01 04:00:00+05:45,2025-07-01 04:45:00+05:45,23.859673,16.167559,23.752083,65.747619,3319.800588,2025-07-01 04:00:00+05:45,2025-06-30 22:15:00+00:00,0.0,...,180,860.8,25.0,0.0,0.0,0.0,96,3,19,94


In [ ]:
# ==========================================
# MASTER DATASET VALIDATION
# ==========================================

print("MASTER DATASET VALIDATION")
print("=" * 60)

# Dataset dimensions
print("\nDataset Shape:", master_df.shape)

# Column names
print("\nColumns:")
print(master_df.columns.tolist())

# Data types
print("\nData Types:")
display(master_df.dtypes.to_frame("Data Type"))

# Missing values
print("\nMissing Values:")
missing_report = pd.DataFrame({
    "Missing Count": master_df.isnull().sum(),
    "Missing Percentage": (
        master_df.isnull().mean() * 100
    ).round(2)
})

display(missing_report)

# Duplicate timestamps
print(
    "\nDuplicate standardized timestamps:",
    master_df["Datetime"].duplicated().sum()
)

# Chronological order
print(
    "Chronologically sorted:",
    master_df["Datetime"].is_monotonic_increasing
)

# Timestamp coverage
print("\nTimestamp Coverage:")
print("First:", master_df["Datetime"].min())
print("Last:", master_df["Datetime"].max())

MASTER DATASET VALIDATION

Dataset Shape: (10540, 22)

Columns:
['Datetime', 'datetime_openaq_original', 'PM2.5', 'PM1', 'Temperature', 'Relative_Humidity', 'Particle_Count_0.3um', 'datetime_weather_original', 'datetime_utc', 'precipitation', 'rain', 'wind_speed_10m', 'wind_direction_10m', 'surface_pressure', 'boundary_layer_height', 'shortwave_radiation', 'direct_radiation', 'diffuse_radiation', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high']

Data Types:


,Data Type
Datetime,"datetime64[us, Asia/Kathmandu]"
datetime_openaq_original,str
PM2.5,float64
PM1,float64
Temperature,float64
Relative_Humidity,float64
Particle_Count_0.3um,float64
datetime_weather_original,str
datetime_utc,str
precipitation,float64



Missing Values:


,Missing Count,Missing Percentage
Datetime,0,0.00
datetime_openaq_original,0,0.00
PM2.5,0,0.00
PM1,0,0.00
Temperature,0,0.00
Relative_Humidity,0,0.00
Particle_Count_0.3um,0,0.00
datetime_weather_original,0,0.00
datetime_utc,0,0.00
precipitation,0,0.00



Duplicate standardized timestamps: 0
Chronologically sorted: True

Timestamp Coverage:
First: 2025-07-01 00:00:00+05:45
Last: 2026-09-30 23:00:00+05:45


In [ ]:
# Create a working copy of the successfully merged dataset
final_master_df = master_df.copy()

# Rename columns
column_mapping = {
    "Datetime": "datetime",
    "datetime_openaq_original": "openaq_timestamp",
    "datetime_weather_original": "openmeteo_timestamp",
    "PM2.5": "pm2_5",
    "PM1": "pm1",
    "Temperature": "temperature",
    "Relative_Humidity": "relative_humidity",
    "Particle_Count_0.3um": "particle_count_0_3um"
}

final_master_df.rename(columns=column_mapping, inplace=True)

print("Column names standardized successfully.")
print(final_master_df.columns.tolist())

Column names standardized successfully.
['datetime', 'openaq_timestamp', 'pm2_5', 'pm1', 'temperature', 'relative_humidity', 'particle_count_0_3um', 'openmeteo_timestamp', 'datetime_utc', 'precipitation', 'rain', 'wind_speed_10m', 'wind_direction_10m', 'surface_pressure', 'boundary_layer_height', 'shortwave_radiation', 'direct_radiation', 'diffuse_radiation', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high']


In [ ]:
TIMEZONE = "Asia/Kathmandu"

# Main standardized hourly timestamp
final_master_df["datetime"] = (
    pd.to_datetime(final_master_df["datetime"], utc=True)
    .dt.tz_convert(TIMEZONE)
)

# Original OpenAQ timestamp
final_master_df["openaq_timestamp"] = (
    pd.to_datetime(final_master_df["openaq_timestamp"], utc=True)
    .dt.tz_convert(TIMEZONE)
)

# Original Open-Meteo local timestamp
final_master_df["openmeteo_timestamp"] = (
    pd.to_datetime(final_master_df["openmeteo_timestamp"], utc=True)
    .dt.tz_convert(TIMEZONE)
)

# Open-Meteo UTC timestamp
final_master_df["datetime_utc"] = pd.to_datetime(
    final_master_df["datetime_utc"], utc=True
)

print("Datetime columns converted successfully.")
print(final_master_df[
    ["datetime", "openaq_timestamp",
     "openmeteo_timestamp", "datetime_utc"]
].dtypes)

Datetime columns converted successfully.
datetime               datetime64[us, Asia/Kathmandu]
openaq_timestamp       datetime64[us, Asia/Kathmandu]
openmeteo_timestamp    datetime64[us, Asia/Kathmandu]
datetime_utc                      datetime64[us, UTC]
dtype: object


In [ ]:
print("=" * 65)
print("FINAL MASTER DATASET VALIDATION")
print("=" * 65)

print("\nDataset Shape:", final_master_df.shape)

print("\nColumn Names:")
print(final_master_df.columns.tolist())

print("\nData Types:")
print(final_master_df.dtypes)

print("\nMissing Values:")
missing_report = pd.DataFrame({
    "Missing Count": final_master_df.isnull().sum(),
    "Missing Percentage": (
        final_master_df.isnull().mean() * 100
    ).round(2)
})

print(missing_report[missing_report["Missing Count"] > 0])

print("\nDuplicate Timestamps:",
      final_master_df["datetime"].duplicated().sum())

print("Chronologically Sorted:",
      final_master_df["datetime"].is_monotonic_increasing)

print("\nTimestamp Coverage:")
print("First:", final_master_df["datetime"].min())
print("Last:", final_master_df["datetime"].max())

# Verify the original timestamp alignment
alignment_difference = (
    final_master_df["openaq_timestamp"]
    - final_master_df["openmeteo_timestamp"]
)

print("\nOpenAQ - Open-Meteo Timestamp Difference:")
print(alignment_difference.value_counts().head())

print("\n" + "=" * 65)

FINAL MASTER DATASET VALIDATION

Dataset Shape: (10540, 22)

Column Names:
['datetime', 'openaq_timestamp', 'pm2_5', 'pm1', 'temperature', 'relative_humidity', 'particle_count_0_3um', 'openmeteo_timestamp', 'datetime_utc', 'precipitation', 'rain', 'wind_speed_10m', 'wind_direction_10m', 'surface_pressure', 'boundary_layer_height', 'shortwave_radiation', 'direct_radiation', 'diffuse_radiation', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high']

Data Types:
datetime                 datetime64[us, Asia/Kathmandu]
openaq_timestamp         datetime64[us, Asia/Kathmandu]
pm2_5                                           float64
pm1                                             float64
temperature                                     float64
relative_humidity                               float64
particle_count_0_3um                            float64
openmeteo_timestamp      datetime64[us, Asia/Kathmandu]
datetime_utc                        datetime64[us, UTC]
precipitation

In [ ]:
MASTER_OUTPUT = DATASET_DIR / "master_dataset_hadigaun.csv"

final_master_df.to_csv(MASTER_OUTPUT, index=False)

print("Final master dataset saved successfully:")
print(MASTER_OUTPUT.resolve())

Final master dataset saved successfully:
D:\MSDSA\4th Sem\Thesis\Dataset\master_dataset_hadigaun.csv


In [ ]:
# Create a clean copy
final_master_df = master_df.copy()

# Standardize column names
column_mapping = {
    "Datetime": "datetime",
    "PM2.5": "pm2_5",
    "PM1": "pm1",
    "Temperature": "temperature",
    "Relative_Humidity": "relative_humidity",
    "Particle_Count_0.3um": "particle_count_0_3um"
}

final_master_df.rename(columns=column_mapping, inplace=True)

# Retain only the standardized timestamp and analytical variables
final_columns = [
    "datetime",
    "pm2_5",
    "pm1",
    "temperature",
    "relative_humidity",
    "particle_count_0_3um",
    "precipitation",
    "rain",
    "wind_speed_10m",
    "wind_direction_10m",
    "surface_pressure",
    "boundary_layer_height",
    "shortwave_radiation",
    "direct_radiation",
    "diffuse_radiation",
    "cloud_cover",
    "cloud_cover_low",
    "cloud_cover_mid",
    "cloud_cover_high"
]

final_master_df = final_master_df[final_columns].copy()

print("Final dataset shape:", final_master_df.shape)
print(final_master_df.head())

Final dataset shape: (10540, 19)
                   datetime      pm2_5        pm1  temperature  \
0 2025-07-01 00:00:00+05:45  13.717708   9.164286    24.237649   
1 2025-07-01 01:00:00+05:45  11.052827   7.255308    23.981399   
2 2025-07-01 02:00:00+05:45   9.482589   6.263244    23.723066   
3 2025-07-01 03:00:00+05:45  10.492857   6.811310    23.914285   
4 2025-07-01 04:00:00+05:45  23.859673  16.167559    23.752083   

   relative_humidity  particle_count_0_3um  precipitation  rain  \
0          64.782738           1849.218770            0.4   0.4   
1          65.276935           1480.463806            0.2   0.2   
2          66.015773           1320.891368            0.5   0.5   
3          65.384673           1410.139893            0.0   0.0   
4          65.747619           3319.800588            0.0   0.0   

   wind_speed_10m  wind_direction_10m  surface_pressure  \
0            1.00                 174             861.8   
1            1.32                 169            

In [ ]:
# Validate final dataset
assert final_master_df.columns[0] == "datetime"
assert final_master_df.shape == (10540, 19)
assert final_master_df["datetime"].is_unique
assert final_master_df["datetime"].is_monotonic_increasing

print("Final Dataset Shape:", final_master_df.shape)
print("Duplicate Timestamps:", final_master_df["datetime"].duplicated().sum())
print("Chronologically Sorted:", final_master_df["datetime"].is_monotonic_increasing)

print("\nMissing Values:")
print(final_master_df.isnull().sum()[final_master_df.isnull().sum() > 0])

# Save final master dataset
MASTER_OUTPUT = DATASET_DIR / "master_dataset_hadigaun.csv"
final_master_df.to_csv(MASTER_OUTPUT, index=False)

print("\nFinal master dataset saved:", MASTER_OUTPUT.resolve())

Final Dataset Shape: (10540, 19)
Duplicate Timestamps: 0
Chronologically Sorted: True

Missing Values:
boundary_layer_height    477
dtype: int64

Final master dataset saved: D:\MSDSA\4th Sem\Thesis\Dataset\master_dataset_hadigaun.csv


This completes the Data Acquisition step. Further data verification, cleaning, and analysis is done in the EDA steps, and is available in the EDA Notebook.